# PA Traffic Observations — Matching rows only

Loads the hourly `pa_traffic_observations.log-YYYYMMDDHH` files from `H:/Observation2.0/pa_traffic_observations/`, parses only the *Matching rows* section in each file, and builds **`source_data_matches`**: one row per session that hit an observation.

Each row is tagged with `source_hour` (from the filename) and `source_file` so you can group, filter, and pivot easily. Duplicate-count tables in the same files are ignored.

After cleanup, rows are left-joined to **`Splunk_TC_Address_Index.csv`** on `matched_ip` = `indicator` to produce **`matches_tc_df`**, then **sorted by `matched_ip`** (string order on the normalized address).

In [1]:
import os
import re
import glob
from io import StringIO
from datetime import datetime

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 60)

LOG_DIR = r'Z:\ACD-Files\Meleik Hyman\pa_traffic_observations'
files = sorted(glob.glob(os.path.join(LOG_DIR, 'pa_traffic_observations.log-*')))
print(f'Found {len(files)} files')
print('First:', os.path.basename(files[0]))
print('Last: ', os.path.basename(files[-1]))

Found 574 files
First: pa_traffic_observations.log-2026050510
Last:  pa_traffic_observations.log-2026052909


## Parser

Each file has a `Duplicate counts:` section followed by `Matching rows:`. We locate `Matching rows:` and parse only that fixed-width table, then tag rows with `source_hour` (from the filename suffix) and `source_file`.

In [2]:
HOUR_RE = re.compile(r'log-(\d{10})$')


def hour_from_filename(path):
    m = HOUR_RE.search(os.path.basename(path))
    return datetime.strptime(m.group(1), '%Y%m%d%H') if m else None


def parse_file(path):
    with open(path, 'r', encoding='utf-8', errors='replace') as f:
        text = f.read()

    if 'Matching rows:' in text:
        _, match_part = text.split('Matching rows:', 1)
    else:
        match_part = ''

    match_df = pd.DataFrame()
    match_lines = [ln for ln in match_part.splitlines() if ln.strip()]
    if match_lines:
        buf = StringIO('\n'.join(match_lines))
        try:
            match_df = pd.read_fwf(buf)
            if str(match_df.columns[0]).lower().startswith('unnamed'):
                match_df = match_df.drop(columns=match_df.columns[0])
        except Exception as e:
            print(f'  ! could not parse matching rows in {os.path.basename(path)}: {e}')

    return match_df


_m = parse_file(files[0])
print('Sample file:', os.path.basename(files[0]))
print('  matching rows:', len(_m))
_m

Sample file: pa_traffic_observations.log-2026050510
  matching rows: 48


,Unnamed: 1,timestamp,session_id,port_number,application,action,matching_ip,matched_column,matched_ip,total_bytes,sent_bytes,received_bytes,total_packets,packets_sent,packets_received,reason_for_session_end
0,2026/05/05,09:13:29,6191117,47759,incomplete,allow,193.163.125.51,src_ip,193.163.125.51,62,62,0,1,1,0,aged-out
1,2026/05/05,09:13:29,1149322950,41415,ssdp,allow,172.202.118.67,src_ip,172.202.118.67,141,141,0,1,1,0,aged-out
2,2026/05/05,09:13:29,1074243713,33108,incomplete,allow,193.163.125.52,src_ip,193.163.125.52,60,60,0,1,1,0,aged-out
3,2026/05/05,09:13:28,1110873964,37310,incomplete,allow,193.163.125.31,src_ip,193.163.125.31,60,60,0,1,1,0,aged-out
4,2026/05/05,09:13:28,1111004683,48793,incomplete,allow,91.191.209.46,src_ip,91.191.209.46,60,60,0,1,1,0,aged-out
5,2026/05/05,09:13:29,69700276,46767,incomplete,allow,89.190.156.34,src_ip,89.190.156.34,60,60,0,1,1,0,aged-out
6,2026/05/05,09:13:29,69012354,44685,incomplete,allow,193.163.125.253,src_ip,193.163.125.253,60,60,0,1,1,0,aged-out
7,2026/05/05,09:13:29,1081853426,48793,incomplete,allow,91.191.209.46,src_ip,91.191.209.46,60,60,0,1,1,0,aged-out
8,2026/05/05,09:13:29,1079639054,34173,incomplete,allow,193.163.125.43,src_ip,193.163.125.43,60,60,0,1,1,0,aged-out
9,2026/05/05,09:13:29,6602612,42986,incomplete,allow,193.163.125.95,src_ip,193.163.125.95,62,62,0,1,1,0,aged-out


In [3]:
_m.columns.tolist()

['Unnamed: 1',
 'timestamp',
 'session_id',
 'port_number',
 'application',
 'action',
 'matching_ip',
 'matched_column',
 'matched_ip',
 'total_bytes',
 'sent_bytes',
 'received_bytes',
 'total_packets',
 'packets_sent',
 'packets_received',
 'reason_for_session_end']

## Load all files

Loops through every hourly file and stacks matching rows only. Each row gets `source_hour` (a real `Timestamp`) and `source_file`.

In [4]:
match_frames = []

for i, path in enumerate(files, 1):
    hour = hour_from_filename(path)
    fname = os.path.basename(path)
    m = parse_file(path)
    if not m.empty:
        m.insert(0, 'source_hour', hour)
        m.insert(1, 'source_file', fname)
        match_frames.append(m)
    if i % 25 == 0 or i == len(files):
        print(f'  parsed {i}/{len(files)}')

source_data_matches = pd.concat(match_frames, ignore_index=True) if match_frames else pd.DataFrame()

print()
print(f'source_data_matches: {len(source_data_matches):,} rows  x  {source_data_matches.shape[1]} cols')

  parsed 25/574
  parsed 50/574
  parsed 75/574
  parsed 100/574
  parsed 125/574
  parsed 150/574
  parsed 175/574
  parsed 200/574
  parsed 225/574
  parsed 250/574
  parsed 275/574
  parsed 300/574
  parsed 325/574
  parsed 350/574
  parsed 375/574
  parsed 400/574
  parsed 425/574
  parsed 450/574
  parsed 475/574
  parsed 500/574
  parsed 525/574
  parsed 550/574
  parsed 574/574

source_data_matches: 85,036 rows  x  18 cols


In [5]:
pd.DataFrame({'matched_ip': source_data_matches['matched_ip'].astype(str).str.strip().unique()})

,matched_ip
0,193.163.125.51
1,172.202.118.67
2,193.163.125.52
3,193.163.125.31
4,91.191.209.46
...,...
459,141.98.11.82
460,91.191.209.74
461,146.70.45.166
462,164.90.217.247


In [6]:
import glob

# Pull all files from 05/05/2026 to 05/29/2026
obs_dir = r"Z:\HTOC\Data_Analytics\Data\OpDiv_Observations"
opdiv_obs_files = sorted(
    glob.glob(obs_dir + r"\htoc_opdiv_obs_d2026*.csv")
)

# Filter for files between 2026-05-05 and 2026-05-29
from datetime import datetime

start_date = datetime.strptime("20260505", "%Y%m%d")
end_date = datetime.strptime("20260529", "%Y%m%d")

filtered_files = [
    f for f in opdiv_obs_files
    if start_date <= datetime.strptime(f.split('_d')[-1][:8], "%Y%m%d") <= end_date
]

# Load and combine all the selected files
TC_obs_df = (
    pd.concat([pd.read_csv(f) for f in filtered_files], ignore_index=True)
    .drop_duplicates(subset=['indicator', 'obs_date'])
)
TC_obs_df

,indicator,API_UserName,obs_date,OpDiv,indicator_key,observations,curr_date
0,102.68.77.161,35664937356778434955,2026-05-05,HRSA,102.68.77.161_HRSA,3,2026-05-05
1,103.120.116.162,15920309593055310684,2026-05-05,CMS,103.120.116.162_CMS,24,2026-05-05
3,103.125.163.10,35664937356778434955,2026-05-05,HRSA,103.125.163.10_HRSA,7,2026-05-05
4,103.125.233.22,35664937356778434955,2026-05-05,HRSA,103.125.233.22_HRSA,3,2026-05-05
5,103.138.146.172,50189120947314147395,2026-05-05,NIH,103.138.146.172_NIH,1,2026-05-05
...,...,...,...,...,...,...,...
68098,websitedemos.net,20790633968691748718,2026-05-29,DHA,websitedemos.net_DHA,26,2026-05-29
68100,whatistheurl.com,20790633968691748718,2026-05-29,DHA,whatistheurl.com_DHA,23,2026-05-29
68101,wvmetronews.com,15920309593055310684,2026-05-29,CMS,wvmetronews.com_CMS,4,2026-05-29
68104,www.deepseek.com,20790633968691748718,2026-05-29,DHA,www.deepseek.com_DHA,6,2026-05-29


## SOURCE vs TC — same-day alignment

For each calendar day and each indicator (IP):

| Scenario | SOURCE (log) | TC |
|----------|--------------|-----|
| `aligned_same_day` | X | X — observed the same day in both |
| `source_only_not_in_tc` | X | — in log that day, not in TC that day |
| `tc_only_not_in_source` | — | X — in TC that day, not in log that day |

Output **`source_tc_presence`**: one row per indicator and date, with `X` in `source` and/or `ThreatConnect` when present that day. Limited to indicators in **`TC_Splunk_Address_Index_20260505.csv`**.

Writes in this folder:

- **`TC_SOURCE_OBS_ANALYSIS.csv`** — full combined table
- **`TC_SOURCE_OBS_ANALYSIS.xlsx`** — sheet **`All`** plus one sheet per calendar day (`2026-05-05`, …); daily sheets omit the `date` column

In [ ]:
# SOURCE vs TC — same-day indicator alignment
# For each calendar day: was each IP seen in the PA log (SOURCE) also recorded in TC that day, and vice versa?

DATE_FMT = '%Y-%m-%d'
COMPARE_START = '2026-05-05'
COMPARE_END = '2026-05-29'
TC_INDEX_PATH = r'H:\HTOC\notebooks\Gap Observation 2.0\TC_Splunk_Address_Index_20260505.csv'


def _normalize_ip(s):
    return s.astype(str).str.strip()


tc_index_indicators = set(
    _normalize_ip(pd.read_csv(TC_INDEX_PATH)['indicator']).dropna()
)


def _source_obs_dates(df):
    """Calendar date for each log row (session date, else log-file hour)."""
    ts = pd.Series(pd.NaT, index=df.index)
    unnamed = [c for c in df.columns if str(c).lower().startswith('unnamed')]
    if unnamed and 'timestamp' in df.columns:
        combined = (
            df[unnamed[0]].astype(str).str.strip()
            + ' '
            + df['timestamp'].astype(str).str.strip()
        )
        ts = pd.to_datetime(combined, format='%Y/%m/%d %H:%M:%S', errors='coerce')
    elif 'timestamp' in df.columns:
        ts = pd.to_datetime(df['timestamp'], errors='coerce')
    if ts.isna().all() and 'source_hour' in df.columns:
        ts = pd.to_datetime(df['source_hour'], errors='coerce')
    return ts.dt.strftime(DATE_FMT)


# Unique (date, indicator) seen in SOURCE logs
source_daily = (
    source_data_matches.assign(
        obs_date=_source_obs_dates(source_data_matches),
        indicator=_normalize_ip(source_data_matches['matched_ip']),
    )
    .dropna(subset=['obs_date', 'indicator'])
    .query('indicator != ""')
    .drop_duplicates(subset=['obs_date', 'indicator'])
    [['obs_date', 'indicator']]
    .assign(in_source=True)
)

# Unique (date, indicator) recorded in TC
tc_daily = (
    TC_obs_df.assign(
        obs_date=pd.to_datetime(TC_obs_df['obs_date'], errors='coerce').dt.strftime(DATE_FMT),
        indicator=_normalize_ip(TC_obs_df['indicator']),
    )
    .dropna(subset=['obs_date', 'indicator'])
    .query('indicator != ""')
    .drop_duplicates(subset=['obs_date', 'indicator'])
    [['obs_date', 'indicator']]
    .assign(in_tc=True)
)

PRESENT = 'X'

# Full outer join: every indicator-day pair from either side
_merged = source_daily.merge(
    tc_daily, on=['obs_date', 'indicator'], how='outer'
).fillna({'in_source': False, 'in_tc': False})

_merged['in_source'] = _merged['in_source'].astype(bool)
_merged['in_tc'] = _merged['in_tc'].astype(bool)

mask = (
    (_merged['obs_date'] >= COMPARE_START)
    & (_merged['obs_date'] <= COMPARE_END)
    & (_merged['indicator'].isin(tc_index_indicators))
)

# Final output: indicator, date, presence markers (X = seen that day)
source_tc_presence = (
    _merged.loc[mask]
    .assign(
        source=lambda d: d['in_source'].map({True: PRESENT, False: ''}),
        ThreatConnect=lambda d: d['in_tc'].map({True: PRESENT, False: ''}),
    )
    .rename(columns={'obs_date': 'date'})
    [['Indicator', 'Date', 'Source', 'ThreatConnect']]
    .sort_values(['date', 'indicator'])
    .reset_index(drop=True)
)

from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

_HEADER_FILL = PatternFill(start_color='366092', end_color='366092', fill_type='solid')
_HEADER_FONT = Font(bold=True, color='FFFFFF')
_CENTER_COLS = {'source', 'ThreatConnect'}


def _format_tc_source_obs_worksheet(ws):
    """Header row, auto column width, freeze panes; center X marker columns."""
    for cell in ws[1]:
        cell.fill = _HEADER_FILL
        cell.font = _HEADER_FONT
        cell.alignment = Alignment(horizontal='center', vertical='center')
    ws.freeze_panes = 'A2'

    for col_idx, column_cells in enumerate(ws.iter_cols(min_row=1, max_row=ws.max_row), start=1):
        header = str(column_cells[0].value or '')
        max_len = max(len(str(cell.value or '')) for cell in column_cells)
        ws.column_dimensions[get_column_letter(col_idx)].width = min(max(max_len, len(header)) + 2, 45)
        if header in _CENTER_COLS:
            for cell in column_cells[1:]:
                cell.alignment = Alignment(horizontal='center', vertical='center')


TC_SOURCE_OBS_DIR = r'H:\HTOC\notebooks\Gap Observation 2.0'
TC_SOURCE_OBS_PATH = os.path.join(TC_SOURCE_OBS_DIR, 'TC_SOURCE_OBS_ANALYSIS.csv')
TC_SOURCE_OBS_XLSX = os.path.join(TC_SOURCE_OBS_DIR, 'TC_SOURCE_OBS_ANALYSIS.xlsx')

source_tc_presence.to_csv(TC_SOURCE_OBS_PATH, index=False)

with pd.ExcelWriter(TC_SOURCE_OBS_XLSX, engine='openpyxl') as writer:
    source_tc_presence.to_excel(writer, sheet_name='All', index=False)
    for obs_date, day_df in source_tc_presence.groupby('date', sort=True):
        day_df.drop(columns=['date']).to_excel(writer, sheet_name=obs_date, index=False)
    for ws in writer.book.worksheets:
        _format_tc_source_obs_worksheet(ws)

n_dates = source_tc_presence['date'].nunique()
print(f'{len(source_tc_presence):,} indicator-day rows ({len(tc_index_indicators):,} index indicators)')
print(f'Wrote {TC_SOURCE_OBS_PATH}')
print(f'Wrote {TC_SOURCE_OBS_XLSX} (All + {n_dates} date sheets)')
source_tc_presence

C:\Users\jaskew\AppData\Local\Temp\ipykernel_15652\616522499.py:68: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ).fillna({'in_source': False, 'in_tc': False})
